# Retreiver Demonstration

In [2]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F

/work/mbouthil/.conda/envs/myuwenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split='train')

100%|██████████| 8841823/8841823 [01:18<00:00, 113127.96it/s]


# Retreiver Demonstration

### We begin by loading the Vector Database:

**Recall:** this vector database is constructed using the trained Passage Encoder

In [1]:
model = 'base_tuned_10k'

In [3]:
# Loading Index
index = faiss.read_index(f"/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_{model}.index")
print(f"Index contains {index.ntotal} vectors")

Index contains 8841823 vectors


### Load the trained Query Encoder

In [4]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    f"/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_{model}"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ) #.to("cuda")

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 178.73it/s, Materializing param=pooler.dense.weight]                               


### Embedding the Quesetion

In [6]:
question = queries['8']
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
# print('Embedding length: ', q_emb.shape[1])

 In humans, the normal set point for body temperature is 


In [7]:
K = 10
scores, ids = index.search(q_emb, K)
scores = scores[0]
ids = ids[0]

In [8]:
for i, id in enumerate(ids):
    print(scores[i])
    print(corpus[str(id)]['text'])
    print("\n\n")

0.66961026
Humans can only survive within a very narrow range of temperatures-like other mammals and birds they have to keep their body temperature constant at all times, around 37Â°C. Just five degrees too high or low would be a dangerous condition. In healthy individuals, the body temperature (oral temperature) is somewhere between 36,5 and 37,5. It slightly increases during the day since the morning (from 6:00 a.m.).



0.6679701
Normal Body Temperature is Highly Variable Keep in mind, this number, whether itâs 98.6 or 98.2 degrees F, is just an average. The human body has a wide range that is considered normal and each individual may have their own normal range.



0.6623548
1 What is the normal temperature for a human? Dr. Chevies Newman Dr. Newman. 100.4 is considered: A clinically relevant fever, anything above that is an issue, normal body temp is 98.6 and is pretty consistent even in pregnancy.



0.6589227
Human Body Temperature. Human body temperature is generally consider

### It Works! 